# Data Mining Assignment — Apache Spark Data Lake Preprocessing

**Name:** Aryan Kadian
**Roll Number:** LCS2023051
**Dataset:** `sales.csv`
**Environment:** Google Colab + PySpark

This notebook builds a simple two-layer Data Lake:

```
data_lake/
|-- bronze/
|   `-- sales.csv              <- raw data, kept exactly as provided
`-- silver/
    `-- preprocessed_sales/    <- Spark output after basic preprocessing
```

All discrepancy counts shown below are computed with Spark operations at run time — nothing is hard-coded.

## 0. Setup — install PySpark

In [1]:
!pip install -q pyspark

## 1. Data Lake Structure (Bronze / Silver)

The raw `sales.csv` is copied **byte-for-byte** into the Bronze layer. If the file is not already present in the
runtime (e.g. a fresh Colab session), it is downloaded from the class repository. An MD5 checksum confirms the Bronze
copy is identical to the source.

In [2]:
import os
import shutil
import hashlib
import urllib.request

DATA_LAKE_DIR = "data_lake"
BRONZE_DIR = os.path.join(DATA_LAKE_DIR, "bronze")
SILVER_DIR = os.path.join(DATA_LAKE_DIR, "silver")
SILVER_OUTPUT_DIR = os.path.join(SILVER_DIR, "preprocessed_sales")
BRONZE_FILE = os.path.join(BRONZE_DIR, "sales.csv")

DATASET_URL = ("https://raw.githubusercontent.com/necromancersanju-netizen/"
               "data-mining-data-lake-assignment/main/dataset/sales.csv")

os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(SILVER_OUTPUT_DIR, exist_ok=True)


def md5sum(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


# Locate the provided dataset (uploaded to Colab, or inside a cloned repo); otherwise download it.
candidates = ["sales.csv", "/content/sales.csv", "dataset/sales.csv", "../../dataset/sales.csv"]
source_file = next((p for p in candidates if os.path.isfile(p)), None)
if source_file is None:
    source_file = "sales.csv"
    urllib.request.urlretrieve(DATASET_URL, source_file)
    print("Downloaded dataset from the class repository.")

# Copy into Bronze without any modification
if os.path.abspath(source_file) != os.path.abspath(BRONZE_FILE):
    shutil.copyfile(source_file, BRONZE_FILE)

BRONZE_MD5 = md5sum(BRONZE_FILE)
assert md5sum(source_file) == BRONZE_MD5, "Bronze copy differs from the source file!"
print("Source file :", source_file)
print("Bronze file :", BRONZE_FILE)
print("Bronze MD5  :", BRONZE_MD5, "(identical to source)")

Downloaded dataset from the class repository.
Source file : sales.csv
Bronze file : data_lake/bronze/sales.csv
Bronze MD5  : 2b7d6737635600be42aa639140cd6ea9 (identical to source)


In [3]:
# Show the Data Lake folder tree
for root, dirs, files in sorted(os.walk(DATA_LAKE_DIR)):
    depth = root.count(os.sep)
    print("    " * depth + os.path.basename(root) + "/")
    for f in sorted(files):
        print("    " * (depth + 1) + f)

data_lake/
    bronze/
        sales.csv
    silver/
        preprocessed_sales/


## 2. Load and Inspect with Spark

The Bronze file is read with **every column as a string** (`inferSchema=False`). This keeps the raw values intact so
that malformed numbers and dates can be detected instead of being silently turned into nulls during loading.

ANSI mode is disabled and the `CORRECTED` time-parser policy is used so that invalid casts / dates return `null`
consistently on both Spark 3.x and Spark 4.x.

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("SalesDataLakePreprocessing")
         .master("local[*]")
         .config("spark.sql.ansi.enabled", "false")
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 20:40:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.3


In [5]:
bronze_df = (spark.read
             .option("header", True)
             .option("inferSchema", False)
             .csv(BRONZE_FILE))

bronze_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [6]:
bronze_df.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

In [7]:
raw_count = bronze_df.count()
print("Total records in Bronze:", raw_count)
print("Number of columns      :", len(bronze_df.columns))

Total records in Bronze: 1000
Number of columns      : 13


### 2.1 Identify Data Discrepancies

#### (a) Duplicates

In [8]:
distinct_count = bronze_df.distinct().count()
exact_duplicates = raw_count - distinct_count
print("Exact duplicate rows:", exact_duplicates)

# Show the rows that appear more than once
(bronze_df.groupBy(bronze_df.columns).count()
 .where(F.col("count") > 1)
 .orderBy("order_id")
 .show(truncate=False))

Exact duplicate rows: 10


+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100055  |C0270      |Pranav Iyer     |Dry Fruits Pack  |Grocery        |7       |930       |12              |Cash on Delivery|Coimbatore|Tamil Nadu    |2026-08-20|Delivered   |2    |
|100145  |C0007      |Sai Mishra      |Face Wash        |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka     |2025-08-28|Delivered   |2    |
|100297  |C0149      |Manish Pillai   |Dry Fruits Pack  |Grocery        |7      

In [9]:
# order_id values that are reused by *different* records (not exact duplicates)
dup_order_ids = (bronze_df.distinct()
                 .groupBy("order_id").count()
                 .where(F.col("count") > 1))
print("order_id values shared by different records:", dup_order_ids.count())
(bronze_df.distinct()
 .join(dup_order_ids.select("order_id"), "order_id")
 .orderBy("order_id")
 .show(6, truncate=False))

order_id values shared by different records: 8


+--------+-----------+--------------+----------------------+---------------+--------+----------+----------------+--------------+---------+---------+----------+------------+
|order_id|customer_id|customer_name |product               |category       |quantity|unit_price|discount_percent|payment_method|city     |state    |order_date|order_status|
+--------+-----------+--------------+----------------------+---------------+--------+----------+----------------+--------------+---------+---------+----------+------------+
|100064  |C0149      |Manish Pillai |Basmati Rice 5kg      |Grocery        |1       |560       |40              |Debit Card    |Jaipur   |Rajasthan|2026-08-31|Delivered   |
|100064  |C0276      |Ananya Agarwal|Microwave Oven        |Home Appliances|3       |16610     |0               |UPI           |Hyderabad|Telangana|2025-05-15|Delivered   |
|100072  |C0211      |Meera Chopra  |Competitive Exam Guide|Books          |9       |770       |20              |Credit Card   |Jaipur 

#### (b) Null / missing values

In [10]:
# A value counts as missing if it is null or only whitespace
null_counts = bronze_df.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in bronze_df.columns
])
null_counts.show(truncate=False)

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



#### (c) Text-format inconsistencies
Checks for leading/trailing or repeated spaces, and for values that differ only in capitalization
(e.g. `beauty` vs `Beauty`, `DELHI` vs `Delhi`).

In [11]:
TEXT_COLS = ["customer_id", "customer_name", "product", "category",
             "payment_method", "city", "state", "order_status"]

space_issues = bronze_df.select([
    F.sum(F.when((F.col(c) != F.trim(F.col(c))) | F.col(c).rlike("\\s{2,}"), 1).otherwise(0)).alias(c)
    for c in TEXT_COLS
])
print("Rows with extra spaces per column:")
space_issues.show(truncate=False)

# Values whose spelling differs only by case / surrounding spaces from another value
case_variants = []
for c in TEXT_COLS:
    n_raw = bronze_df.select(F.trim(F.col(c))).where(F.col(c).isNotNull()).distinct().count()
    n_norm = bronze_df.select(F.lower(F.trim(F.col(c)))).where(F.col(c).isNotNull()).distinct().count()
    case_variants.append((c, n_raw, n_norm, n_raw - n_norm))
spark.createDataFrame(case_variants, ["column", "distinct_trimmed", "distinct_lowercase", "extra_case_variants"]).show()

Rows with extra spaces per column:


+-----------+-------------+-------+--------+--------------+----+-----+------------+
|customer_id|customer_name|product|category|payment_method|city|state|order_status|
+-----------+-------------+-------+--------+--------------+----+-----+------------+
|0          |3            |0      |0       |0             |3   |0    |0           |
+-----------+-------------+-------+--------+--------------+----+-----+------------+



+--------------+----------------+------------------+-------------------+
|        column|distinct_trimmed|distinct_lowercase|extra_case_variants|
+--------------+----------------+------------------+-------------------+
|   customer_id|             335|               335|                  0|
| customer_name|             302|               302|                  0|
|       product|              57|                57|                  0|
|      category|              13|                 8|                  5|
|payment_method|               5|                 5|                  0|
|          city|              16|                11|                  5|
|         state|               9|                 9|                  0|
|  order_status|               5|                 5|                  0|
+--------------+----------------+------------------+-------------------+



In [12]:
# Look at the inconsistent spellings in category and city
for c in ["category", "city"]:
    (bronze_df.groupBy(c).count()
     .orderBy(F.lower(F.trim(F.col(c))), c)
     .show(30, truncate=False))

+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |136  |
|beauty         |2    |
|Books          |128  |
|CLOTHING       |1    |
|Clothing       |108  |
|ELECTRONICS    |1    |
|Electronics    |103  |
|Furniture      |132  |
|Grocery        |136  |
|Home Appliances|115  |
|SPORTS         |1    |
|Sports         |136  |
|sports         |1    |
+---------------+-----+

+-----------+-----+
|city       |count|
+-----------+-----+
|NULL       |8    |
|Bengaluru  |91   |
|Chennai    |88   |
|Coimbatore |81   |
|coimbatore |1    |
|DELHI      |2    |
|Delhi      |81   |
|delhi      |1    |
| Hyderabad |2    |
|HYDERABAD  |1    |
|Hyderabad  |94   |
|JAIPUR     |1    |
|Jaipur     |84   |
|Kolkata    |81   |
|Lucknow    |90   |
| Mumbai    |1    |
|Mumbai     |106  |
|Pune       |82   |
|Vijayawada |105  |
+-----------+-----+



In [13]:
# customer_name values that are not in "Title Case" or have extra spaces
(bronze_df.where((F.col("customer_name") != F.initcap(F.trim(F.col("customer_name")))))
 .select("order_id", F.concat(F.lit("'"), F.col("customer_name"), F.lit("'")).alias("customer_name_raw"))
 .show(truncate=False))

+--------+------------------+
|order_id|customer_name_raw |
+--------+------------------+
|100156  |'Meera Joshi '    |
|100295  |'Akash Jain '     |
|100886  |'Tanvi Chaudhary '|
+--------+------------------+



#### (d) Invalid numerical values
- `quantity` must be a positive whole number.
- `unit_price` must be greater than 0.
- `discount_percent` must lie between 0 and 100.

In [14]:
num_df = bronze_df.select(
    "order_id", "quantity", "unit_price", "discount_percent",
    F.col("quantity").cast("int").alias("qty_num"),
    F.col("unit_price").cast("double").alias("price_num"),
    F.col("discount_percent").cast("double").alias("disc_num"),
)

invalid_quantity = (F.col("quantity").isNotNull() & (F.col("qty_num").isNull() | (F.col("qty_num") <= 0)))
invalid_price = (F.col("unit_price").isNotNull() & (F.col("price_num").isNull() | (F.col("price_num") <= 0)))
invalid_discount = (F.col("discount_percent").isNotNull() &
                    (F.col("disc_num").isNull() | (F.col("disc_num") < 0) | (F.col("disc_num") > 100)))

num_df.select(
    F.sum(F.when(invalid_quantity, 1).otherwise(0)).alias("invalid_quantity"),
    F.sum(F.when(invalid_price, 1).otherwise(0)).alias("invalid_unit_price"),
    F.sum(F.when(invalid_discount, 1).otherwise(0)).alias("invalid_discount_percent"),
).show()

num_df.describe("qty_num", "price_num", "disc_num").show()

+----------------+------------------+------------------------+
|invalid_quantity|invalid_unit_price|invalid_discount_percent|
+----------------+------------------+------------------------+
|              12|                 8|                       8|
+----------------+------------------+------------------------+



+-------+------------------+------------------+------------------+
|summary|           qty_num|         price_num|          disc_num|
+-------+------------------+------------------+------------------+
|  count|              1000|              1000|              1000|
|   mean|             5.355|            5702.4|            19.948|
| stddev|2.9916425128890194|10990.194484849431|14.236782711763993|
|    min|                -2|            -500.0|              -5.0|
|    max|                10|           81010.0|             120.0|
+-------+------------------+------------------+------------------+



In [15]:
for label, cond, col in [("quantity", invalid_quantity, "quantity"),
                         ("unit_price", invalid_price, "unit_price"),
                         ("discount_percent", invalid_discount, "discount_percent")]:
    print(f"Invalid {label} values:")
    num_df.where(cond).groupBy(col).count().orderBy(col).show()

Invalid quantity values:


+--------+-----+
|quantity|count|
+--------+-----+
|      -1|    1|
|      -2|    5|
|       0|    6|
+--------+-----+

Invalid unit_price values:


+----------+-----+
|unit_price|count|
+----------+-----+
|      -250|    1|
|      -500|    3|
|         0|    4|
+----------+-----+

Invalid discount_percent values:


+----------------+-----+
|discount_percent|count|
+----------------+-----+
|              -5|    4|
|             120|    4|
+----------------+-----+



#### (e) Inconsistent / invalid dates
The expected format is `yyyy-MM-dd`. Values that do not match it are listed, and each is tested against a few other
common formats to see whether it is a *valid date in a different format* or a *clearly invalid date*.

In [16]:
STANDARD_DATE_FMT = "yyyy-MM-dd"
ALTERNATE_DATE_FMTS = ["yyyy/MM/dd", "dd/MM/yyyy", "MM-dd-yyyy"]

standard_date = F.to_date(F.trim(F.col("order_date")), STANDARD_DATE_FMT)
parsed_date = F.coalesce(standard_date,
                         *[F.to_date(F.trim(F.col("order_date")), fmt) for fmt in ALTERNATE_DATE_FMTS])

date_check = bronze_df.select(
    "order_id", "order_date",
    standard_date.alias("standard_parse"),
    parsed_date.alias("any_format_parse"),
)

date_check.select(
    F.sum(F.when(F.col("order_date").isNull(), 1).otherwise(0)).alias("missing_dates"),
    F.sum(F.when(F.col("order_date").isNotNull() & F.col("standard_parse").isNull(), 1).otherwise(0))
     .alias("non_standard_or_invalid"),
    F.sum(F.when(F.col("standard_parse").isNull() & F.col("any_format_parse").isNotNull(), 1).otherwise(0))
     .alias("valid_but_other_format"),
    F.sum(F.when(F.col("order_date").isNotNull() & F.col("any_format_parse").isNull(), 1).otherwise(0))
     .alias("clearly_invalid"),
).show()

date_check.where(F.col("standard_parse").isNull()).orderBy("order_id").show(truncate=False)

date_check.select(F.min("standard_parse").alias("earliest_date"),
                  F.max("standard_parse").alias("latest_date")).show()

+-------------+-----------------------+----------------------+---------------+
|missing_dates|non_standard_or_invalid|valid_but_other_format|clearly_invalid|
+-------------+-----------------------+----------------------+---------------+
|            0|                      6|                     2|              4|
+-------------+-----------------------+----------------------+---------------+

+--------+----------+--------------+----------------+
|order_id|order_date|standard_parse|any_format_parse|
+--------+----------+--------------+----------------+
|100097  |31/02/2026|NULL          |NULL            |
|100358  |2025/07/14|NULL          |2025-07-14      |
|100448  |not-a-date|NULL          |NULL            |
|100634  |2025-02-30|NULL          |NULL            |
|100637  |2026-13-05|NULL          |NULL            |
|100969  |09-15-2026|NULL          |2026-09-15      |
+--------+----------+--------------+----------------+



+-------------+-----------+
|earliest_date|latest_date|
+-------------+-----------+
|   2025-01-01| 2026-09-30|
+-------------+-----------+



## 3. Basic Preprocessing — Silver Layer

### Step 1 — Remove exact duplicate rows

In [17]:
dedup_df = bronze_df.dropDuplicates()
after_dedup_count = dedup_df.count()
print("Rows before:", raw_count)
print("Rows after removing exact duplicates:", after_dedup_count)
print("Removed:", raw_count - after_dedup_count)

Rows before: 1000
Rows after removing exact duplicates: 990
Removed: 10


### Step 2 — Trim spaces and standardize capitalization in text fields
- All text columns: trim leading/trailing spaces and collapse repeated inner spaces.
- `customer_name`, `category`, `city`, `state`, `order_status` → Title Case (`initcap`).
- `customer_id` → upper case.
- `product` and `payment_method` are only trimmed, since values such as `USB-C Hub`, `UPI` and `Cash on Delivery` are
  already consistent and `initcap` would distort them (the case-variant check above found none in these columns).

In [18]:
def clean_spaces(col_name):
    trimmed = F.trim(F.regexp_replace(F.col(col_name), "\\s+", " "))
    return F.when(trimmed == "", None).otherwise(trimmed)   # blank strings become proper nulls

TITLE_CASE_COLS = ["customer_name", "category", "city", "state", "order_status"]

text_df = dedup_df
for c in TEXT_COLS:
    text_df = text_df.withColumn(c, clean_spaces(c))
for c in TITLE_CASE_COLS:
    text_df = text_df.withColumn(c, F.initcap(F.col(c)))
text_df = text_df.withColumn("customer_id", F.upper(F.col("customer_id")))

for c in ["category", "city"]:
    text_df.groupBy(c).count().orderBy(c).show(truncate=False)

+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |137  |
|Books          |127  |
|Clothing       |108  |
|Electronics    |104  |
|Furniture      |129  |
|Grocery        |134  |
|Home Appliances|113  |
|Sports         |138  |
+---------------+-----+



+----------+-----+
|city      |count|
+----------+-----+
|NULL      |8    |
|Bengaluru |90   |
|Chennai   |88   |
|Coimbatore|81   |
|Delhi     |83   |
|Hyderabad |97   |
|Jaipur    |83   |
|Kolkata   |81   |
|Lucknow   |89   |
|Mumbai    |105  |
|Pune      |81   |
|Vijayawada|104  |
+----------+-----+



### Step 3 — Handle missing values
- `customer_name`, `payment_method`, `city` are descriptive fields; missing values are replaced with `"Unknown"` so
  the rest of the order (product, amount, date) is preserved.
- A missing `city` is filled from the `state` only when that state maps to exactly **one** city in the data
  (e.g. `Telangana → Hyderabad`); otherwise it becomes `"Unknown"`.

In [19]:
# state -> city lookup, only for states that have a single known city
single_city_states = (text_df.where(F.col("city").isNotNull())
                      .groupBy("state")
                      .agg(F.countDistinct("city").alias("n_cities"), F.first("city").alias("state_city"))
                      .where(F.col("n_cities") == 1)
                      .select("state", "state_city"))
single_city_states.orderBy("state").show(truncate=False)

filled_df = (text_df.join(single_city_states, on="state", how="left")
             .withColumn("city", F.coalesce(F.col("city"), F.col("state_city")))
             .drop("state_city"))

filled_df = filled_df.fillna({"customer_name": "Unknown",
                              "payment_method": "Unknown",
                              "city": "Unknown"})

filled_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in TEXT_COLS
]).show()

+--------------+----------+
|state         |state_city|
+--------------+----------+
|Andhra Pradesh|Vijayawada|
|Delhi         |Delhi     |
|Karnataka     |Bengaluru |
|Rajasthan     |Jaipur    |
|Telangana     |Hyderabad |
|Uttar Pradesh |Lucknow   |
|West Bengal   |Kolkata   |
+--------------+----------+



+-----------+-------------+-------+--------+--------------+----+-----+------------+
|customer_id|customer_name|product|category|payment_method|city|state|order_status|
+-----------+-------------+-------+--------+--------------+----+-----+------------+
|          0|            0|      0|       0|             0|   0|    0|           0|
+-----------+-------------+-------+--------+--------------+----+-----+------------+



### Step 4 — Validate `quantity`, `unit_price`, `discount_percent`
Values are cast to numeric types. Clearly invalid values (quantity ≤ 0, unit price ≤ 0, discount outside 0–100) are
set to `null` rather than deleting the whole order, so the rest of the record remains usable. Each affected row is
marked in a `dq_issues` column (added in Step 6).

In [20]:
typed_df = (filled_df
            .withColumn("quantity_raw", F.col("quantity"))
            .withColumn("unit_price_raw", F.col("unit_price"))
            .withColumn("discount_raw", F.col("discount_percent"))
            .withColumn("quantity", F.col("quantity").cast("int"))
            .withColumn("unit_price", F.col("unit_price").cast("double"))
            .withColumn("discount_percent", F.col("discount_percent").cast("double")))

typed_df = (typed_df
            .withColumn("invalid_quantity",
                        F.col("quantity_raw").isNotNull() & (F.col("quantity").isNull() | (F.col("quantity") <= 0)))
            .withColumn("invalid_unit_price",
                        F.col("unit_price_raw").isNotNull() & (F.col("unit_price").isNull() | (F.col("unit_price") <= 0)))
            .withColumn("invalid_discount",
                        F.col("discount_raw").isNotNull() & (F.col("discount_percent").isNull() |
                                                             (F.col("discount_percent") < 0) |
                                                             (F.col("discount_percent") > 100))))

typed_df = (typed_df
            .withColumn("quantity", F.when(F.col("invalid_quantity"), None).otherwise(F.col("quantity")))
            .withColumn("unit_price", F.when(F.col("invalid_unit_price"), None).otherwise(F.col("unit_price")))
            .withColumn("discount_percent",
                        F.when(F.col("invalid_discount"), None).otherwise(F.col("discount_percent"))))

typed_df.select(
    F.sum(F.col("invalid_quantity").cast("int")).alias("quantity_set_to_null"),
    F.sum(F.col("invalid_unit_price").cast("int")).alias("unit_price_set_to_null"),
    F.sum(F.col("invalid_discount").cast("int")).alias("discount_set_to_null"),
).show()

+--------------------+----------------------+--------------------+
|quantity_set_to_null|unit_price_set_to_null|discount_set_to_null|
+--------------------+----------------------+--------------------+
|                  12|                     8|                   8|
+--------------------+----------------------+--------------------+



### Step 5 — Convert dates to one consistent format
Dates written in another recognizable format (`yyyy/MM/dd`, `dd/MM/yyyy`, `MM-dd-yyyy`) are converted to a proper
Spark `DateType` stored as `yyyy-MM-dd`. Clearly invalid values (e.g. text, or an impossible calendar date) become
`null` and are flagged.

In [21]:
dated_df = (typed_df
            .withColumn("order_date_raw", F.col("order_date"))
            .withColumn("order_date", parsed_date)
            .withColumn("invalid_date", F.col("order_date_raw").isNotNull() & F.col("order_date").isNull())
            .withColumn("reformatted_date",
                        F.col("order_date").isNotNull() & (F.trim(F.col("order_date_raw")) !=
                                                           F.date_format("order_date", STANDARD_DATE_FMT))))

(dated_df.where(F.col("invalid_date") | F.col("reformatted_date"))
 .select("order_id", "order_date_raw", "order_date", "invalid_date")
 .orderBy("order_id")
 .show(truncate=False))

+--------+--------------+----------+------------+
|order_id|order_date_raw|order_date|invalid_date|
+--------+--------------+----------+------------+
|100097  |31/02/2026    |NULL      |true        |
|100358  |2025/07/14    |2025-07-14|false       |
|100448  |not-a-date    |NULL      |true        |
|100634  |2025-02-30    |NULL      |true        |
|100637  |2026-13-05    |NULL      |true        |
|100969  |09-15-2026    |2026-09-15|false       |
+--------+--------------+----------+------------+



### Step 6 — Flag remaining issues, keep useful records, and finalize the schema
- A `dq_issues` column records what was fixed/flagged for each row (e.g. `invalid_quantity;invalid_date`), so records
  are preserved but downstream users can still filter them.
- `order_id` values reused by **different** orders are kept (they look like distinct transactions) and flagged as
  `duplicate_order_id`.
- A final `dropDuplicates()` removes rows that became identical only after text standardization.

In [22]:
dup_id_window = Window.partitionBy("order_id")
flagged_df = dated_df.withColumn("order_id_count", F.count(F.lit(1)).over(dup_id_window))

dq_issues = F.concat_ws(";",
    F.when(F.col("order_id_count") > 1, F.lit("duplicate_order_id")),
    F.when(F.col("invalid_quantity"), F.lit("invalid_quantity")),
    F.when(F.col("invalid_unit_price"), F.lit("invalid_unit_price")),
    F.when(F.col("invalid_discount"), F.lit("invalid_discount")),
    F.when(F.col("invalid_date"), F.lit("invalid_date")),
)

FINAL_COLS = ["order_id", "customer_id", "customer_name", "product", "category",
              "quantity", "unit_price", "discount_percent", "payment_method",
              "city", "state", "order_date", "order_status", "dq_issues"]

silver_df = (flagged_df
             .withColumn("order_id", F.col("order_id").cast("int"))
             .withColumn("dq_issues", F.when(dq_issues == "", None).otherwise(dq_issues))
             .select(FINAL_COLS)
             .dropDuplicates()
             .orderBy("order_id", "customer_id"))

silver_count = silver_df.count()
print("Rows removed by final dropDuplicates:", after_dedup_count - silver_count)
silver_df.printSchema()
silver_df.show(10, truncate=False)

Rows removed by final dropDuplicates: 0
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- dq_issues: string (nullable = true)



+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+---------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|dq_issues|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+---------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710.0    |20.0            |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |NULL     |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680.0     |5.0             |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |NULL     |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6    

In [23]:
silver_df.groupBy("dq_issues").count().orderBy(F.desc("count")).show(truncate=False)

+------------------+-----+
|dq_issues         |count|
+------------------+-----+
|NULL              |942  |
|duplicate_order_id|16   |
|invalid_quantity  |12   |
|invalid_discount  |8    |
|invalid_unit_price|8    |
|invalid_date      |4    |
+------------------+-----+



### Save to the Silver layer
The DataFrame is written with Spark as CSV (single partition) to `data_lake/silver/preprocessed_sales/`.
Spark always writes a folder of part files, so the single part file is also copied to
`data_lake/silver/sales_silver.csv` for easy submission.

In [24]:
(silver_df.coalesce(1)
 .write.mode("overwrite")
 .option("header", True)
 .option("dateFormat", "yyyy-MM-dd")
 .csv(SILVER_OUTPUT_DIR))

part_files = [f for f in os.listdir(SILVER_OUTPUT_DIR) if f.startswith("part-") and f.endswith(".csv")]
SILVER_CSV = os.path.join(SILVER_DIR, "sales_silver.csv")
shutil.copyfile(os.path.join(SILVER_OUTPUT_DIR, part_files[0]), SILVER_CSV)

print("Spark output folder:", SILVER_OUTPUT_DIR, "->", sorted(os.listdir(SILVER_OUTPUT_DIR)))
print("Single-file copy   :", SILVER_CSV)

Spark output folder: data_lake/silver/preprocessed_sales -> ['._SUCCESS.crc', '.part-00000-f789c1b4-8f52-4594-aa6d-eca42389eb92-c000.csv.crc', '_SUCCESS', 'part-00000-f789c1b4-8f52-4594-aa6d-eca42389eb92-c000.csv']
Single-file copy   : data_lake/silver/sales_silver.csv


## 4. Verification

In [25]:
print(f"Row count before preprocessing (Bronze): {raw_count}")
print(f"Row count after  preprocessing (Silver): {silver_count}")
print(f"Rows removed (duplicates only)         : {raw_count - silver_count}")

Row count before preprocessing (Bronze): 1000
Row count after  preprocessing (Silver): 990
Rows removed (duplicates only)         : 10


In [26]:
silver_read_df = (spark.read
                  .option("header", True)
                  .option("inferSchema", True)
                  .csv(SILVER_OUTPUT_DIR))

print("Rows read back from Silver:", silver_read_df.count())
silver_read_df.printSchema()
silver_read_df.show(10, truncate=False)

Rows read back from Silver: 990
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- dq_issues: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+---------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|dq_issues|
+--------

In [27]:
# Re-run the discrepancy checks on the Silver data
print("Exact duplicate rows in Silver:", silver_read_df.count() - silver_read_df.distinct().count())

checks = silver_read_df.select(
    F.sum(F.when(F.col("quantity") <= 0, 1).otherwise(0)).alias("quantity_le_0"),
    F.sum(F.when(F.col("unit_price") <= 0, 1).otherwise(0)).alias("unit_price_le_0"),
    F.sum(F.when((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100), 1).otherwise(0))
     .alias("discount_out_of_range"),
    F.sum(F.when(F.col("customer_name").isNull() | F.col("payment_method").isNull() | F.col("city").isNull(), 1)
          .otherwise(0)).alias("missing_text_fields"),
    F.sum(F.when(sum([(F.col(c) != F.trim(F.col(c))).cast("int")
                      for c in TEXT_COLS]) > 0, 1).otherwise(0)).alias("rows_with_extra_spaces"),
)
checks.show()

for c in ["category", "city", "payment_method"]:
    silver_read_df.groupBy(c).count().orderBy(c).show(truncate=False)

Exact duplicate rows in Silver: 0


+-------------+---------------+---------------------+-------------------+----------------------+
|quantity_le_0|unit_price_le_0|discount_out_of_range|missing_text_fields|rows_with_extra_spaces|
+-------------+---------------+---------------------+-------------------+----------------------+
|            0|              0|                    0|                  0|                     0|
+-------------+---------------+---------------------+-------------------+----------------------+



+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |137  |
|Books          |127  |
|Clothing       |108  |
|Electronics    |104  |
|Furniture      |129  |
|Grocery        |134  |
|Home Appliances|113  |
|Sports         |138  |
+---------------+-----+



+----------+-----+
|city      |count|
+----------+-----+
|Bengaluru |93   |
|Chennai   |88   |
|Coimbatore|81   |
|Delhi     |84   |
|Hyderabad |98   |
|Jaipur    |84   |
|Kolkata   |81   |
|Lucknow   |89   |
|Mumbai    |105  |
|Pune      |81   |
|Unknown   |1    |
|Vijayawada|105  |
+----------+-----+

+----------------+-----+
|payment_method  |count|
+----------------+-----+
|Cash on Delivery|206  |
|Credit Card     |203  |
|Debit Card      |199  |
|Net Banking     |204  |
|UPI             |170  |
|Unknown         |8    |
+----------------+-----+



In [28]:
# Confirm the Bronze file was never modified
assert md5sum(BRONZE_FILE) == BRONZE_MD5
print("Bronze file unchanged. MD5:", md5sum(BRONZE_FILE))

for root, dirs, files in sorted(os.walk(DATA_LAKE_DIR)):
    depth = root.count(os.sep)
    print("    " * depth + os.path.basename(root) + "/")
    for f in sorted(files):
        print("    " * (depth + 1) + f)

Bronze file unchanged. MD5: 2b7d6737635600be42aa639140cd6ea9
data_lake/
    bronze/
        sales.csv
    silver/
        sales_silver.csv
        preprocessed_sales/
            ._SUCCESS.crc
            .part-00000-f789c1b4-8f52-4594-aa6d-eca42389eb92-c000.csv.crc
            _SUCCESS
            part-00000-f789c1b4-8f52-4594-aa6d-eca42389eb92-c000.csv


### Summary of preprocessing performed

1. **Bronze layer** — raw `sales.csv` copied unchanged (MD5 verified before and after processing).
2. **Exact duplicates** — removed with `dropDuplicates()` (and again after text standardization).
3. **Text cleanup** — trimmed leading/trailing spaces, collapsed repeated spaces, converted blank strings to null;
   standardized capitalization (`initcap`) for `customer_name`, `category`, `city`, `state`, `order_status`,
   upper-cased `customer_id`.
4. **Missing values** — `customer_name` and `payment_method` filled with `"Unknown"`; `city` filled from `state`
   when that state has a single city in the data, otherwise `"Unknown"`.
5. **Numeric validation** — `quantity` → int, `unit_price` / `discount_percent` → double; quantity ≤ 0,
   unit price ≤ 0 and discount outside 0–100 set to null (record kept).
6. **Dates** — parsed to `DateType` in `yyyy-MM-dd`; alternate formats (`yyyy/MM/dd`, `dd/MM/yyyy`, `MM-dd-yyyy`)
   converted; impossible/unparseable dates set to null.
7. **Record preservation** — no rows dropped except duplicates; a `dq_issues` column flags rows with invalid values
   or re-used `order_id`s.
8. **Silver layer** — saved as CSV in `data_lake/silver/preprocessed_sales/` (plus `sales_silver.csv` copy) and
   read back with Spark for verification.

In [29]:
spark.stop()